In [1]:
"""
exp02 Step 01: Mechanism Check (PROTOCOL.md §11.2; EXPLORATION ON 2005-2014 ONLY)

Question: before any model is trained, does the stop-and-re-entry mechanism leave room for information to beat
buy-and-hold?

Variants, each simulated over the whole exploration period as ONE continuous window (start invested, like buy-and-hold):
    - stop + fixed-delay re-entry (1, 5, 20 cash decisions): what the stop costs or gains without any information;
    - stop + ORACLE re-entry: re-enters at the lowest 15:59 open among the decisions allowed before the forced re-entry.
      It looks into the future: it is the UPPER BOUND of what perfect re-entry timing could add with this stop;
    - the 200-session moving-average rule (no stop), the textbook trend rule.

How to read it:
    - the ORACLE is a very loose ceiling (perfect foresight over up to 60 sessions): it only rules a k out if even the
      oracle fails;
    - the more useful evidence is the POST-EXIT PATH (last cells): the mean log share gain of re-entering n decisions
      after each stop, for n = 1 .. 60, against the same quantity after ANY session (the drift reference);
    - share_gain_pct per episode = shares owned after re-entry / shares owned before the exit - 1 (before fees):
      > 0 means the strategy bought back lower than it sold.

2005-2014 is training data in every walk-forward fold (the first validation quarter starts in 2015), so looking at it
does not spend any validation or test window. One exploration entry per variant is written to the trial log.
Prerequisite: the raw data only. Outputs: store04_experiments/exp02_stop_reentry/step01_mechanism_check_data/.
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT PLOTLY
import plotly.express as px
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp02_stop_reentry import config as exp_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION FUNCTIONS
from so.core.reentry_simulation import simulate_stop_reentry_dict, get_buy_and_hold_result_dict, simulate_trend_rule_dict, \
                                       get_fixed_delay_reentry_func, get_oracle_reentry_func, get_window_session_idx_arr, get_post_exit_path_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_mechanism_check_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "exploration")

Experiment:	exp02_stop_reentry	Protocol:	1.0	Config Hash:	53a86cd30f
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp02_stop_reentry/step01_mechanism_check_data/
ℹ️ No earlier 'exploration' entry of exp02_stop_reentry in the trial log (config hash 53a86cd30f).


In [4]:
# CALL FUNCTION TO READ THE COMPLETE MINUTE BARS (RAW FOLDER OF so.paths)
complete_ohlcv_pdf = get_complete_ohlcv_pdf()
# CALL FUNCTION TO CONVERT THE COMPLETE DATA TO ARRAYS (USED BY THE SIMULATOR)
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
# CALL FUNCTION TO BUILD THE DAILY FEATURES AND TARGET (SECONDS; pipeline/step06 SAVES THE SAME TABLE AS A RECORD)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
# DISPLAY INFORMATION
print(f"min timestamp:\t{complete_ohlcv_pdf.timestamp.min()}")
print(f"max timestamp:\t{complete_ohlcv_pdf.timestamp.max()}")
print(f"Sessions:\t{len(daily_pdf):,}\tLabelled:\t{int(daily_pdf['y_fwd_positive'].notna().sum()):,}")

min timestamp:	2005-01-03 09:30:00-05:00
max timestamp:	2026-08-13 15:59:00-04:00
Sessions:	5,436	Labelled:	5,416


In [5]:
# DEFINE THE EXPLORATION WINDOW (SHARED RESEARCH DISCIPLINE: 2005-01-03 -> 2014-12-31)
exploration_start_str, exploration_end_str = config.EXPLORATION_START_DATE_STR, config.EXPLORATION_END_DATE_STR
# COLLECT THE WINDOW SESSIONS AND THE STOP VOLATILITY
exploration_session_idx_arr = get_window_session_idx_arr(ohlcv_array_dict, exploration_start_str, exploration_end_str)
volatility_arr = daily_pdf["daily_volatility"].to_numpy(dtype=float)
# CALL FUNCTION TO SIMULATE BUY AND HOLD
buy_hold_dict = get_buy_and_hold_result_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str)
# DISPLAY INFORMATION
print(f"Exploration:\t{exploration_start_str} -> {exploration_end_str}\t({len(exploration_session_idx_arr):,} sessions)")
print(f"Buy & hold:\ttotal return {buy_hold_dict['metric_dict']['total_return']:.2%} | Sharpe {buy_hold_dict['metric_dict']['sharpe_ratio']:.2f} | max drawdown {buy_hold_dict['metric_dict']['max_drawdown']:.2%}")

Exploration:	2005-01-03 -> 2014-12-31	(2,516 sessions)
Buy & hold:	total return 69.02% | Sharpe 0.36 | max drawdown -56.44%


In [6]:
# DICTIONARIES TO HOLD THE SIMULATIONS AND THE METRIC ROWS
simulation_dict_dict, metric_dict_list = {}, []
# ITERATE OVER THE STOP MULTIPLIERS
for stop_k in exp_config.STOP_K_LIST:
    # DEFINE THE RE-ENTRY VARIANTS OF THIS STOP
    variant_dict = {**{f"delay_{delay}": (get_fixed_delay_reentry_func(delay), "delay") for delay in exp_config.FIXED_DELAY_SESSION_LIST},
                    "oracle": (get_oracle_reentry_func(ohlcv_array_dict, exploration_session_idx_arr, exp_config.MAX_CASH_SESSIONS), "oracle")}
    # ITERATE OVER THE VARIANTS
    for variant_str, (reentry_func, reason_str) in variant_dict.items():
        # SIMULATE THE VARIANT
        simulation_dict = simulate_stop_reentry_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str, volatility_arr, stop_k, reentry_func, reason_str,
                                                     max_cash_sessions_in=exp_config.MAX_CASH_SESSIONS)
        simulation_dict_dict[(stop_k, variant_str)] = simulation_dict
        # STORE THE METRICS
        metric_dict_list.append({"stop_k": stop_k, "variant": variant_str, **simulation_dict["metric_dict"],
                                 "excess_return": simulation_dict["metric_dict"]["total_return"] - buy_hold_dict["metric_dict"]["total_return"]})
# SIMULATE THE TREND RULE (NO STOP, NO FORCED RE-ENTRY)
trend_dict = simulate_trend_rule_dict(ohlcv_array_dict, daily_pdf, exploration_start_str, exploration_end_str, f"ma{exp_config.TREND_RULE_MA_SESSIONS}_dist_pct")
simulation_dict_dict[(np.nan, "trend_ma200")] = trend_dict
metric_dict_list.append({"stop_k": np.nan, "variant": f"trend_ma{exp_config.TREND_RULE_MA_SESSIONS}", **trend_dict["metric_dict"],
                         "excess_return": trend_dict["metric_dict"]["total_return"] - buy_hold_dict["metric_dict"]["total_return"]})
# ADD BUY AND HOLD
metric_dict_list.append({"stop_k": np.nan, "variant": "buy_hold", **buy_hold_dict["metric_dict"], "excess_return": 0.0})
# CONVERT TO A DATAFRAME
mechanism_pdf = pd.DataFrame(metric_dict_list)
# PREVIEW DATAFRAME
mechanism_pdf[["stop_k", "variant", "total_return", "excess_return", "sharpe_ratio", "max_drawdown", "stop_count", "forced_reentry_count",
               "cash_session_pct", "mean_cash_sessions", "mean_share_gain_pct", "positive_episode_pct"]].round(4)

,stop_k,variant,total_return,excess_return,sharpe_ratio,max_drawdown,stop_count,forced_reentry_count,cash_session_pct,mean_cash_sessions,mean_share_gain_pct,positive_episode_pct
0,3.0,delay_1,0.0647,-0.6254,0.1306,-0.6438,259.0,0.0,0.0004,1.0000,-0.0017,0.4595
1,3.0,delay_5,-0.0607,-0.7509,0.0409,-0.5820,169.0,0.0,0.2688,5.0000,-0.0030,0.4083
2,3.0,delay_20,-0.3094,-0.9996,-0.3062,-0.4063,85.0,0.0,0.6421,20.0000,-0.0096,0.3059
3,3.0,oracle,14.2254,13.5352,2.1995,-0.0926,64.0,0.0,0.5109,21.0781,0.0371,0.7969
4,4.0,delay_1,0.2045,-0.4856,0.1931,-0.6504,150.0,0.0,0.0004,1.0000,-0.0021,0.4067
5,4.0,delay_5,0.6291,-0.0611,0.3697,-0.4152,113.0,0.0,0.1797,5.0000,0.0002,0.4513
6,4.0,delay_20,-0.2183,-0.9084,-0.1512,-0.4095,69.0,0.0,0.5197,20.0000,-0.0104,0.3478
7,4.0,oracle,13.4358,12.7457,2.1626,-0.0858,50.0,0.0,0.4052,21.3800,0.0479,0.8400
8,5.0,delay_1,0.2196,-0.4705,0.1996,-0.6338,100.0,0.0,0.0008,1.0000,-0.0031,0.4700
9,5.0,delay_5,0.2005,-0.4897,0.1918,-0.4569,85.0,0.0,0.1360,5.0000,-0.0035,0.3882


In [7]:
"""
Reading the table:
    - excess_return of delay_1 / delay_5 / delay_20: what the stop does on its own (usually negative: whipsaws).
    - excess_return of oracle: the ceiling for this k. If it is negative, stop here for this k.
    - mean_share_gain_pct and positive_episode_pct: how often and by how much the strategy bought back lower than it sold.
"""
# CREATE A FIGURE OF THE EXCESS RETURN PER VARIANT AND k
plot_pdf = mechanism_pdf[mechanism_pdf["stop_k"].notna()].assign(stop_k=lambda df: df["stop_k"].astype(int).astype(str))
fig = px.bar(plot_pdf, x="variant", y="excess_return", color="stop_k", barmode="group", template="plotly_dark", width=1100, height=450,
             title=f"Excess Return Over Buy & Hold, {exploration_start_str} -> {exploration_end_str} (trend rule: {mechanism_pdf.loc[mechanism_pdf['variant'].str.startswith('trend'), 'excess_return'].iloc[0]:.1%})")
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

In [8]:
# DEFINE THE STOP MULTIPLIER TO PLOT
plot_stop_k = exp_config.STOP_K_LIST[1]
# CREATE A FIGURE OF THE EQUITY CURVES
fig = go.Figure()
fig.add_trace(go.Scatter(x=buy_hold_dict["daily_equity_pdf"]["date"], y=buy_hold_dict["daily_equity_pdf"]["equity"], mode="lines", name="buy & hold"))
for (stop_k, variant_str), simulation_dict in simulation_dict_dict.items():
    if stop_k == plot_stop_k or variant_str.startswith("trend"):
        fig.add_trace(go.Scatter(x=simulation_dict["daily_equity_pdf"]["date"], y=simulation_dict["daily_equity_pdf"]["equity"], mode="lines", name=f"{variant_str} (k={stop_k})"))
fig.update_layout(title=f"Equity Curves (k = {plot_stop_k})", yaxis_title="Equity ($)", template="plotly_dark", width=1100, height=550, font=dict(color="white"))
fig.show()

In [9]:
# FUNCTION: GET THE CALENDAR-YEAR RETURNS OF A DAILY EQUITY CURVE
def get_year_return_series(daily_equity_pdf_in, initial_capital_in=config.INITIAL_CAPITAL):
    # CALCULATE THE DAILY RETURNS (THE FIRST ONE FROM THE STARTING CAPITAL)
    equity_arr = np.concatenate([[initial_capital_in], daily_equity_pdf_in["equity"].to_numpy(dtype=float)])
    return_series = pd.Series(equity_arr[1:] / equity_arr[:-1] - 1, index=pd.to_datetime(daily_equity_pdf_in["date"]))
    # RETURN THE COMPOUNDED RETURN PER YEAR
    return return_series.groupby(return_series.index.year).apply(lambda s: float(np.prod(1 + s) - 1))

# BUILD THE YEARLY RETURN TABLE (k = plot_stop_k, TREND RULE AND BUY AND HOLD)
year_return_pdf = pd.DataFrame({"buy_hold": get_year_return_series(buy_hold_dict["daily_equity_pdf"]),
                                **{variant_str: get_year_return_series(simulation_dict["daily_equity_pdf"])
                                   for (stop_k, variant_str), simulation_dict in simulation_dict_dict.items() if stop_k == plot_stop_k or variant_str.startswith("trend")}})
# PREVIEW DATAFRAME
year_return_pdf.style.format("{:.1%}")

,buy_hold,delay_1,delay_5,delay_20,oracle,trend_ma200
date,,,,,,
2005,2.5%,-2.8%,0.7%,-0.9%,10.6%,0.0%
2006,13.6%,8.3%,5.5%,0.2%,16.6%,7.5%
2007,3.5%,2.2%,2.1%,-0.3%,17.8%,-6.7%
2008,-38.3%,-52.2%,-23.9%,-25.5%,30.5%,0.0%
2009,23.5%,29.1%,37.3%,2.5%,69.4%,11.6%
2010,12.9%,14.3%,11.2%,-0.3%,44.0%,-2.8%
2011,-0.2%,-0.3%,-12.4%,0.2%,27.7%,-9.8%
2012,13.4%,11.9%,7.2%,5.1%,24.5%,9.9%
2013,29.5%,28.0%,20.6%,1.0%,38.9%,29.5%


In [10]:
# PREVIEW THE EPISODES OF THE ORACLE AND OF THE NEXT-DAY RE-ENTRY (k = plot_stop_k)
for variant_str in ["delay_1", "oracle"]:
    episode_pdf = simulation_dict_dict[(plot_stop_k, variant_str)]["episode_pdf"]
    print(f"\n{variant_str}: {len(episode_pdf)} episodes | mean share gain {episode_pdf['share_gain_pct'].mean():.2%} | bought back lower in {(episode_pdf['share_gain_pct'] > 0).mean():.0%}")
    display(episode_pdf[["exit_ts", "exit_fill_price", "reentry_ts", "reentry_fill_price", "cash_session_count", "reentry_reason", "share_gain_pct"]].head(15))


delay_1: 150 episodes | mean share gain -0.21% | bought back lower in 41%


,exit_ts,exit_fill_price,reentry_ts,reentry_fill_price,cash_session_count,reentry_reason,share_gain_pct
0,2005-02-03 12:49:00-05:00,118.669601,2005-02-03 15:59:00-05:00,119.06,1,delay,-0.003279
1,2005-02-22 14:15:00-05:00,118.974445,2005-02-22 15:59:00-05:00,118.68,1,delay,0.002481
2,2005-03-16 09:30:00-05:00,119.690000,2005-03-16 15:59:00-05:00,119.21,1,delay,0.004027
3,2005-03-24 15:11:00-05:00,116.652038,2005-03-24 15:59:00-05:00,117.14,1,delay,-0.004166
4,2005-04-14 13:35:00-04:00,116.246568,2005-04-14 15:59:00-04:00,116.26,1,delay,-0.000116
5,2005-06-23 15:55:00-04:00,120.023166,2005-06-23 15:59:00-04:00,120.01,1,delay,0.000110
6,2005-07-07 09:30:00-04:00,118.280000,2005-07-07 15:59:00-04:00,119.82,1,delay,-0.012853
7,2005-08-15 10:12:00-04:00,122.207438,2005-08-15 15:59:00-04:00,123.62,1,delay,-0.011427
8,2005-08-26 10:03:00-04:00,120.907362,2005-08-26 15:59:00-04:00,120.88,1,delay,0.000226
9,2005-09-20 15:04:00-04:00,121.975795,2005-09-20 15:59:00-04:00,122.10,1,delay,-0.001017



oracle: 50 episodes | mean share gain 4.79% | bought back lower in 84%


,exit_ts,exit_fill_price,reentry_ts,reentry_fill_price,cash_session_count,reentry_reason,share_gain_pct
0,2005-02-03 12:49:00-05:00,118.669601,2005-04-20 15:59:00-04:00,113.82,53,oracle,0.042608
1,2005-06-23 15:55:00-04:00,120.023166,2005-06-30 15:59:00-04:00,119.02,6,oracle,0.008429
2,2005-07-07 09:30:00-04:00,118.280000,2005-07-07 15:59:00-04:00,119.82,1,oracle,-0.012853
3,2005-08-15 10:12:00-04:00,122.207438,2005-10-13 15:59:00-04:00,117.83,43,oracle,0.037150
4,2005-11-30 10:08:00-05:00,125.016527,2005-12-30 15:59:00-05:00,124.65,22,oracle,0.002940
5,2006-01-17 09:35:00-05:00,126.221391,2006-02-07 15:59:00-05:00,125.55,16,oracle,0.005348
6,2006-03-08 11:47:00-05:00,127.270009,2006-05-23 15:59:00-04:00,125.97,54,oracle,0.010320
7,2006-06-08 11:19:00-04:00,124.900898,2006-06-13 15:59:00-04:00,122.76,4,oracle,0.017440
8,2006-07-14 11:14:00-04:00,123.324403,2006-07-17 15:59:00-04:00,123.51,2,oracle,-0.001503
9,2006-09-07 10:40:00-04:00,129.515989,2006-09-07 15:59:00-04:00,129.79,1,oracle,-0.002111


In [11]:
"""
Post-exit path: for every stop of the next-day re-entry simulation (it records the most stops), the share gain that
re-entering at the n-th cash decision would have produced. Mean log share gain > 0 at some n = on average prices were
still lower n decisions after the stop (waiting would have paid). The dotted DRIFT REFERENCE is the same quantity for an
exit at ANY session: the period's drift alone. Only the distance between the curves (log_gain_vs_reference) is about stops.
The LOG share gain is averaged because the plain average of S / R - 1 drifts upwards with n even on a random walk
(Jensen's inequality), which would make waiting look better than it is.
"""
# LIST TO HOLD THE PATH SUMMARIES
path_summary_pdf_list = []
# ITERATE OVER THE STOP MULTIPLIERS
for stop_k in exp_config.STOP_K_LIST:
    # CALL FUNCTION TO GET THE POST-EXIT PATHS OF THE NEXT-DAY RE-ENTRY EPISODES
    path_pdf = get_post_exit_path_pdf(ohlcv_array_dict, simulation_dict_dict[(stop_k, "delay_1")]["episode_pdf"], exp_config.MAX_CASH_SESSIONS)
    # SUMMARIZE PER DELAY
    path_summary_pdf_list.append(path_pdf.groupby("cash_decision_count").agg(exit_count=("share_gain_pct", "size"),
                                                                             mean_log_share_gain=("log_share_gain", "mean"),
                                                                             median_share_gain_pct=("share_gain_pct", "median"),
                                                                             gain_share=("share_gain_pct", lambda s: float((s > 0).mean()))).reset_index().assign(stop_k=stop_k))
# CONCATENATE THE SUMMARIES
path_summary_pdf = pd.concat(path_summary_pdf_list, ignore_index=True)
# CALCULATE THE DRIFT REFERENCE: THE SAME LOG GAIN FOR AN EXIT AT ANY SESSION'S 15:59 OPEN (NOT ONLY AFTER STOPS)
fill_open_arr = daily_pdf["fill_open"].to_numpy(dtype=float)[exploration_session_idx_arr]
reference_dict = {n: float(np.mean(np.log(fill_open_arr[:len(fill_open_arr) - n + 1] / fill_open_arr[n - 1:]))) for n in range(1, exp_config.MAX_CASH_SESSIONS + 1)}
path_summary_pdf["drift_reference_log_gain"] = path_summary_pdf["cash_decision_count"].map(reference_dict)
path_summary_pdf["log_gain_vs_reference"] = path_summary_pdf["mean_log_share_gain"] - path_summary_pdf["drift_reference_log_gain"]
# CREATE A FIGURE OF THE MEAN SHARE GAIN PER DELAY
fig = px.line(path_summary_pdf.assign(stop_k=lambda df: df["stop_k"].astype(str)), x="cash_decision_count", y="mean_log_share_gain", color="stop_k", markers=True,
              title="Mean Log Share Gain Of Re-entering n Decisions After A Stop (> 0 = bought back lower)", template="plotly_dark", width=1100, height=450)
fig.add_trace(go.Scatter(x=list(reference_dict.keys()), y=list(reference_dict.values()), mode="lines", name="drift reference (any session)", line=dict(dash="dot", color="white")))
fig.add_hline(y=0, line_dash="dash", line_color="white")
fig.update_layout(yaxis_tickformat=".2%", xaxis_title="cash decisions after the stop (n)")
fig.show()
# PREVIEW THE SUMMARY AT A FEW DELAYS
path_summary_pdf[path_summary_pdf["cash_decision_count"].isin([1, 2, 3, 5, 10, 20, 40, 60])].round(4)

,cash_decision_count,exit_count,mean_log_share_gain,median_share_gain_pct,gain_share,stop_k,drift_reference_log_gain,log_gain_vs_reference
0,1,259,-0.0017,-0.0006,0.4595,3,0.0000,-0.0017
1,2,259,-0.0030,-0.0021,0.3977,3,-0.0002,-0.0028
2,3,259,-0.0040,-0.0022,0.4208,3,-0.0004,-0.0035
4,5,259,-0.0047,-0.0066,0.3900,3,-0.0009,-0.0038
9,10,259,-0.0063,-0.0095,0.3745,3,-0.0020,-0.0043
19,20,259,-0.0068,-0.0130,0.3552,3,-0.0042,-0.0026
39,40,259,-0.0082,-0.0201,0.3591,3,-0.0086,0.0004
59,60,259,-0.0174,-0.0315,0.3089,3,-0.0126,-0.0049
60,1,150,-0.0022,-0.0011,0.4067,4,0.0000,-0.0022
61,2,150,-0.0026,-0.0019,0.4067,4,-0.0002,-0.0024


In [12]:
# SAVE THE MECHANISM CHECK RESULTS
write_csv_file_to_path(mechanism_pdf, f"{output_path_str}mechanism_check_data.csv", "W")
write_csv_file_to_path(year_return_pdf.reset_index().rename(columns={"index": "year"}), f"{output_path_str}mechanism_check_year_return_data.csv", "W")
write_csv_file_to_path(path_summary_pdf, f"{output_path_str}mechanism_check_post_exit_path_data.csv", "W")
# LOG ONE EXPLORATION ENTRY PER VARIANT (NO VALIDATION OR TEST WINDOW IS TOUCHED)
logged_pdf = log_trial_pdf(exp_config, "exploration", mechanism_pdf.assign(window=f"{exploration_start_str}->{exploration_end_str}"), ["stop_k", "variant", "window"],
                           ["total_return", "excess_return", "sharpe_ratio", "max_drawdown", "stop_count", "mean_share_gain_pct"], False,
                           note_str_in="exp02 step01 mechanism check (exploration period only)")
print(f"Logged {len(logged_pdf)} exploration entries (config hash {get_config_hash_str(exp_config)})")

ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp02_stop_reentry/step01_mechanism_check_data/mechanism_check_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp02_stop_reentry/step01_mechanism_check_data/mechanism_check_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp02_stop_reentry/step01_mechanism_check_data/mechanism_check_year_return_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp02_stop_reentry/step01_mechanism_check_data/mechanism_check_year_return_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp02_stop_reentry/step01_mechanism_check_data/mechanis

In [13]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
